# Конспект. Модуль 13: Метрики и мониторинг в production

## 1. Зачем это нужно и как это связано с предыдущими модулями

Это последний **тематический** модуль курса перед капстоуном. Модуль 12 научил нас **обучать** модель на несбалансированных данных; этот модуль закрывает три оставшихся, тесно связанных вопроса полного жизненного цикла модели:

1. **Как честно измерить качество** модели, когда классы сильно несбалансированы (ROC-AUC vs PR-AUC) — то есть не обмануться на этапе валидации так, как чуть не обманулись на этапе обучения в Модуле 12.
2. **Можно ли доверять** самим предсказанным вероятностям как **числам**, а не только как **ранжированию** (калибровка) — критично, поскольку ваша формула Cost-Sensitive Threshold в FraudGuard использует **абсолютное** значение `fraud_probability`, а не просто его порядковое место среди других транзакций.
3. **Что происходит с моделью после деплоя**, когда мир вокруг неё меняется, а переобучение происходит не мгновенно (Data Drift/Concept Drift, PSI) — единственная часть курса про **уже работающую в проде** модель, а не про этап обучения.

## 2. ROC-AUC: формальное определение и почему он может обманывать

### 2.1. Напоминание и вероятностная интерпретация

`TPR = Recall = TP/(TP+FN)`, `FPR = FP/(FP+TN)` (Неделя 4). ROC-кривая — график `TPR` против `FPR` при развёртке порога от `1` до `0`. ROC-AUC — площадь под этой кривой.

**Полезная вероятностная интерпретация ROC-AUC** (пригодится и для интуиции, и для собеседования): ROC-AUC численно равен **вероятности того, что случайно выбранный положительный объект получит от модели более высокий скор, чем случайно выбранный отрицательный объект**. `ROC-AUC=0.5` — модель ранжирует не лучше подбрасывания монетки; `ROC-AUC=1.0` — идеальное ранжирование, вообще никогда не путает порядок.

### 2.2. Почему при сильном дисбалансе ROC-AUC может «врать» — точный численный пример

Возьмём датасет из Модуля 12: `N=10000`, `π=0.002` -> `20` мошеннических, `9980` честных транзакций. Предположим, при некотором пороге модель достигает `TPR=0.90` (поймала `18` из `20` мошеннических), но при этом даёт `FP=500` (пятьсот ложных срабатываний среди честных).

**Посчитаем `FPR`:**

In [ ]:
FPR = FP / (FP+TN) = 500 / 9980 ≈ 0.0501

На графике ROC это точка `(FPR=0.05, TPR=0.90)` — **отличная** точка, близкая к левому верхнему углу (идеалу), она внесёт большой положительный вклад в итоговую площадь под кривой. Если модель ведёт себя похожим образом и на соседних порогах, итоговый **ROC-AUC вполне может составить `0.93` и выше** — выглядит превосходно.

**Теперь посчитаем `Precision` — метрику, которую ROC-AUC вообще не учитывает:**

In [ ]:
Precision = TP / (TP+FP) = 18 / (18+500) = 18/518 ≈ 0.0348

**Precision — всего `3.5%`.** Это означает: из каждых `≈29` транзакций, которые модель помечает как подозрительные, реальным мошенничеством оказывается лишь **одна**. На практике это означает, что служба поддержки/верификации будет завалена ложными тревогами в соотношении `28:1` — с операционной точки зрения такая модель может оказаться **практически бесполезной**, несмотря на впечатляющий `ROC-AUC=0.93`.

### 2.3. Почему так получается — формальная причина

Взгляните на знаменатель `FPR`: `FP+TN`. При сильном дисбалансе `TN` (честных, верно классифицированных транзакций) **огромно** (`9980`-`FP` в нашем примере — всё ещё тысячи). Даже большое **абсолютное** число ложных срабатываний (`500`) остаётся **относительно малой долей** этого огромного пула честных транзакций. `FPR` «прощает» модели абсолютно много ложных тревог, если общий пул честных транзакций достаточно велик — а именно так всегда и бывает при сильном дисбалансе. **Precision**, напротив, сравнивает `FP` **напрямую** с `TP` — числом, которое **обязательно мало** при малочисленном положительном классе — поэтому та же абсолютная величина `FP=500` мгновенно и честно «топит» Precision.

**Это и есть точный, конкретный ответ на первый чек-поинт вопрос модуля.**

## 3. PR-AUC: почему он честнее в этом контексте

`Precision = TP/(TP+FP)` и `Recall = TP/(TP+FN)` — обратите внимание, **ни одна** из этих формул **вообще не использует `TN`**. PR-кривая и площадь под ней (PR-AUC, часто называемая Average Precision) **напрямую** чувствительны к тому, насколько велико число ложных срабатываний **относительно** числа истинных находок, а не относительно всего огромного пула честных объектов. В примере раздела 2.2 та же самая точка (`TPR=Recall=0.90`, `FP=500`) даёт **низкую** Precision (`0.035`) — на PR-кривой эта точка выглядела бы **плохо**, честно отражая операционную бесполезность модели при таком пороге, в отличие от ROC-кривой, где та же точка выглядела превосходно.

**Практический вывод:** для задач с сильным дисбалансом (антифрод, редкие заболевания, поиск дефектов) PR-AUC — более информативная и осторожная метрика по умолчанию; ROC-AUC стоит рассматривать как дополнительную, а не основную метрику.

## 4. Калибровка вероятностей

### 4.1. Ключевой факт, который легко упустить: AUC-метрики нечувствительны к калибровке

И ROC-AUC, и PR-AUC зависят **только** от **относительного порядка** (ранжирования) предсказанных скоров — не от их **абсолютных** значений. Формально: если применить к предсказаниям **любое строго монотонное** преобразование (сохраняющее порядок, но меняющее сами числа), оба AUC **не изменятся вообще**, тогда как содержательный смысл чисел может исказиться до неузнаваемости.

**Численная иллюстрация.** Пусть истинно хорошо откалиброванные вероятности пяти объектов: `[0.1, 0.3, 0.5, 0.7, 0.9]`. Применим монотонное, но «неправильное» преобразование `f(p) = p³`:

In [ ]:
Новые (искажённые) скоры: [0.001, 0.027, 0.125, 0.343, 0.729]

**Порядок объектов полностью сохранился** (первый скор всё ещё наименьший, пятый — наибольший) — значит, ROC-AUC и PR-AUC, посчитанные на этих искажённых числах, будут **абсолютно идентичны** тем же метрикам на исходных, честных вероятностях. Но объект с истинной вероятностью `0.5` теперь **выдаёт себя** за объект с вероятностью всего `0.125` — систематическая, четырёхкратная недооценка. **Если такое искажённое число подставить в формулу Cost-Sensitive Threshold** (`Total Cost = FP×C_check + FN×TransactionAmt`), бизнес-решения окажутся систематически неверными, **несмотря на то, что модель отлично ранжирует объекты и показывает прекрасный AUC**.

**Вывод:** AUC-метрики говорят вам о **качестве ранжирования**, но **ничего** не говорят о том, можно ли **доверять числовым значениям** вероятности напрямую. Для FraudGuard, где вероятность используется в бизнес-формуле, а не только для сортировки, это критично.

### 4.2. Platt Scaling

Простейшее решение: обучить **дополнительную**, отдельную логистическую регрессию **поверх** сырого скора модели (одна переменная — сам скор `F(x)`, один выход — калиброванная вероятность):

In [ ]:
p_calibrated = σ(A·F(x) + B)

Коэффициенты `A`, `B` подбираются методом максимального правдоподобия **на отдельной, не использованной при обучении основной модели выборке** (holdout-калибровочный набор — тот же принцип изоляции train/test, что вы уже многократно применяли). Это, по сути, «пришивание» ещё одной знакомой вам логистической регрессии (Неделя 4) поверх уже готовой модели — простой, параметрический, но не слишком гибкий метод (предполагает, что искажение калибровки имеет именно сигмоидную форму).

### 4.3. Isotonic Regression

Более гибкий, **непараметрический** метод: подбирается произвольная **монотонно неубывающая** ступенчатая функция, отображающая сырой скор в калиброванную вероятность, без предположения о конкретной (сигмоидной) форме искажения. Плата за гибкость — **большая потребность в данных** для надёжной калибровочной выборки: чем больше степеней свободы у метода калибровки, тем легче ему переобучиться на шуме небольшой калибровочной выборки. **Это снова тот же bias-variance компромисс**, который сопровождал весь курс: Platt Scaling — более смещённый (жёсткая сигмоидная форма), но экономный по данным метод; Isotonic Regression — более гибкий (низкий bias), но требовательный к объёму калибровочных данных.

### 4.4. Диагностика: калибровочная кривая и Brier Score

**Калибровочная кривая (reliability diagram):** предсказания разбиваются на бины по значению вероятности (например, `[0.0-0.1)`, `[0.1-0.2)`, ...), и для каждого бина сравнивается **средняя предсказанная** вероятность с **реально наблюдаемой долей** положительного класса в этом бине. Идеально откалиброванная модель даёт точки точно на диагонали `y=x`.

**Brier Score** — скалярная сводная метрика калибровки:

In [ ]:
Brier = (1/N) · Σ(i=1..N) (p_i - y_i)²

**Заметили что-то знакомое?** Это **буквально MSE**, применённый к предсказанной вероятности и бинарной метке — та же самая функция потерь, которую вы вывели с нуля в Модуле 3, только теперь используемая не как целевая функция обучения дерева, а как **метрика оценки качества калибровки** готовой модели.

## 5. Почему Stratified K-Fold обязателен при сильном дисбалансе — точный расчёт риска

Вы уже знаете (Неделя 4), что стратификация сохраняет пропорции классов в каждом фолде. Посчитаем, **насколько велик** реальный риск, если её **не** использовать, на **датасете Модуля 12** (`N=10000`, `20` мошеннических объектов, `K=5` фолдов).

**Приближённая модель риска.** Если бы каждый из `20` мошеннических объектов **независимо** и **равновероятно** попадал в любой из `5` фолдов (упрощение, не учитывающее конечность выборки, но дающее верный порядок величины), вероятность, что **конкретный** мошеннический объект **не попадёт** в конкретный фолд — `4/5=0.8`. Вероятность, что **все 20** мошеннических объектов одновременно **обойдут** этот конкретный фолд стороной:

In [ ]:
0.8^20 ≈ 0.0115   (≈1.15%)

По неравенству объединения (грубая, но полезная верхняя оценка) вероятность, что **хотя бы один** из `5` фолдов останется **вовсе без** мошеннических примеров:

In [ ]:
≤ 5 × 0.0115 ≈ 0.0575   (≈5.75%, то есть примерно 1 случай из 17)

**Это совсем не пренебрежимо малая вероятность.** Фолд без единого положительного примера делает вычисление Recall/Precision/PR-AUC на этом фолде **математически неопределённым** (деление на ноль или бессмысленный результат) — кросс-валидация в таком случае буквально «ломается» на одном из фолдов. Stratified K-Fold полностью устраняет этот риск, принудительно распределяя все `20` положительных примеров **пропорционально** по всем `5` фолдам — этот конкретный численный расчёт (а не просто общая рекомендация «лучше стратифицировать») и есть содержательный ответ на вопрос «почему обязателен» применительно именно к сильному дисбалансу.

## 6. Data Drift vs Concept Drift

### 6.1. Формальное различие

- **Data Drift:** меняется распределение **признаков** `P(X)`, но истинная зависимость `P(Y|X)` **остаётся прежней**. Пример: сезонный рост среднего чека перед праздниками — распределение `TransactionAmt` сдвигается, но то, что **считается** мошенничеством (при данных конкретных значениях признаков), не изменилось.
- **Concept Drift:** меняется сама зависимость `P(Y|X)` — те же самые значения признаков теперь означают **другую** вероятность мошенничества. Пример: мошенники освоили новую схему атаки, которая по «старым» признакам выглядит как обычная, безопасная транзакция.

### 6.2. Почему это различие практически важно

**Data Drift** обычно «самоисправляется» регулярным переобучением на свежих данных — сама закономерность `P(Y|X)`, которую учит модель, не изменилась, изменился лишь состав входящих данных, и переобучение на актуальной выборке это естественно учитывает.

**Concept Drift** значительно опаснее по двум причинам: во-первых, свежих **размеченных** данных, отражающих **новую** схему атаки, может пока быть крайне мало (мошенники только начали её применять). Во-вторых, в реальном антифроде часто присутствует **задержка разметки (label latency)** — подтверждение, что конкретная транзакция была мошеннической, может прийти через недели или месяцы (после жалобы клиента, расследования банка) — пока не накопится достаточно **подтверждённых** новых примеров, модель, обученная на старой концепции, продолжает пропускать новую схему атаки, и ущерб успевает накопиться. Именно поэтому ценен **мониторинг без меток** — детекция **сдвигов во входных данных** (раздел 7), которая может подать сигнал тревоги **до** того, как накопится достаточно размеченных примеров, подтверждающих проблему напрямую.

## 7. PSI (Population Stability Index)

### 7.1. Формула

In [ ]:
PSI = Σ(бины) (actual% - expected%) × ln(actual% / expected%)

где `expected%` — доля объектов в данном бине признака в **эталонном** (обычно — обучающем) распределении, `actual%` — доля в **текущем** (например, сегодняшнем продакшн-трафике) распределении. Признак предварительно разбивается на фиксированные бины (например, по квантилям train-распределения).

**Интерпретация итогового значения:**

In [ ]:
PSI < 0.10          — стабильно, действий не требуется
0.10 ≤ PSI ≤ 0.25    — умеренный сдвиг, начать пристально следить
PSI > 0.25          — серьёзный сдвиг, пересматривать модель / переобучать

### 7.2. Численный пример: считаем PSI вручную

Признак `amt_log`, разбитый на 5 бинов. Эталонное (train) и текущее (продакшн, со сдвигом в сторону более крупных сумм — например, из-за роста среднего чека) распределения:

| Бин | expected% (train) | actual% (продакшн) |
|---|---|---|
| < 3 | 10% | 5% |
| 3–4 | 25% | 15% |
| 4–5 | 30% | 25% |
| 5–6 | 25% | 30% |
| > 6 | 10% | 25% |

**Расчёт по бинам:**

In [ ]:
Бин 1: (0.05-0.10)·ln(0.05/0.10) = (-0.05)·ln(0.5)   = (-0.05)·(-0.6931) = 0.03466
Бин 2: (0.15-0.25)·ln(0.15/0.25) = (-0.10)·ln(0.6)   = (-0.10)·(-0.5108) = 0.05108
Бин 3: (0.25-0.30)·ln(0.25/0.30) = (-0.05)·ln(0.8333) = (-0.05)·(-0.1823) = 0.00912
Бин 4: (0.30-0.25)·ln(0.30/0.25) = ( 0.05)·ln(1.2)    = ( 0.05)·( 0.1823) = 0.00912
Бин 5: (0.25-0.10)·ln(0.25/0.10) = ( 0.15)·ln(2.5)    = ( 0.15)·( 0.9163) = 0.13744

**Сумма:**

In [ ]:
PSI = 0.03466 + 0.05108 + 0.00912 + 0.00912 + 0.13744 ≈ 0.241

**Интерпретация:** `PSI≈0.241` попадает в зону «умеренный сдвиг» (`0.10–0.25`), почти вплотную приближаясь к порогу «серьёзный сдвиг» (`0.25`) — это сигнал **пристально следить** за моделью и, вероятно, готовиться к переобучению в ближайшее время, но ещё не однозначное требование немедленных действий. Обратите внимание, откуда взялся основной вклад в итоговую сумму: **Бин 5** (`0.137` — более половины всего PSI) — именно там сдвиг долей был наибольшим относительно исходной доли (`10%->25%`, рост в 2.5 раза) — PSI, как и Gain в Модуле 6, придаёт больший вес **относительным**, а не только абсолютным изменениям, за счёт логарифмического множителя.

**Прямой ответ на второй чек-поинт вопрос:** `PSI=0.3` — это значение **выше** порога `0.25`, сигнализирующее о **серьёзном** сдвиге распределения. Действие — не просто «следить», а **активно пересматривать модель**: как минимум запланировать переобучение на свежих данных, как максимум — расследовать **причину** сдвига (возможно, это не безобидный Data Drift, а признак Concept Drift, требующий более глубокого анализа, а не просто дообучения на новых данных с теми же старыми метками).

### 7.3. Практическое применение: где это встраивается в архитектуру

PSI обычно считается **регулярно** (например, ежедневно) между эталонным train-распределением и накопленным за последний период продакшн-трафиком — естественная задача для фонового воркера (в вашей архитектуре FraudGuard — Celery Beat, тот же механизм, что уже используется для `recalc_user_features`). Результат логично публиковать через `/health`-эндпоинт (как заложено в спецификации вашего проекта) — вместе с `pr_auc`, `threshold` и `avg_latency_ms`, которые вы уже планировали там выводить.

**Дополнительный практический приём:** PSI можно считать не только по **входным признакам**, но и по **распределению самого выходного скора модели** (`fraud_probability`) — это быстрый «сигнальный маячок»: даже если ни один отдельный признак не сдвинулся заметно, совокупное поведение модели (например, она вдруг стала присваивать заметно больше транзакций к «среднему» уровню риска) может измениться из-за сложных, неочевидных взаимодействий признаков — мониторинг PSI на выходном скоре дополняет мониторинг по отдельным входным признакам, а не заменяет его.

## 8. Практика: код

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score

# --- Часть 1: расхождение ROC-AUC и PR-AUC при сильном дисбалансе ---
X, y = make_classification(n_samples=20000, n_features=25, n_informative=12,
                            weights=[0.998, 0.002], flip_y=0.001, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

model = lgb.LGBMClassifier(n_estimators=300, num_leaves=31,
                            class_weight="balanced", random_state=42, verbose=-1)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)
print(f"ROC-AUC = {roc_auc:.4f}")
print(f"PR-AUC  = {pr_auc:.4f}")
print("Обратите внимание на разрыв между значениями - PR-AUC обычно заметно ниже ROC-AUC "
      "при сильном дисбалансе, честнее отражая операционную сложность задачи.")

# Точка на конкретном пороге - демонстрация раздела 2.2
threshold = 0.3
preds = (proba >= threshold).astype(int)
print(f"При threshold={threshold}: Recall={recall_score(y_test, preds):.4f}, "
      f"Precision={precision_score(y_test, preds, zero_division=0):.4f}")


# --- Часть 2: ручной расчёт PSI между train и искусственно сдвинутым test ---
def calculate_psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10, eps: float = 1e-6) -> float:
    breakpoints = np.quantile(expected, np.linspace(0, 1, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf

    expected_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual)

    expected_pct = np.clip(expected_pct, eps, None)
    actual_pct = np.clip(actual_pct, eps, None)

    return np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

rng = np.random.default_rng(42)
train_amounts = rng.lognormal(mean=4.0, sigma=1.0, size=5000)
# Искусственный сдвиг: "сегодняшний" трафик со средними суммами заметно выше
prod_amounts_shifted = rng.lognormal(mean=4.5, sigma=1.0, size=5000)

psi_value = calculate_psi(train_amounts, prod_amounts_shifted, bins=10)
print(f"\nPSI (train vs искусственно сдвинутый продакшн) = {psi_value:.4f}")

**Что ожидать:** в первой части — разрыв между `ROC-AUC` и `PR-AUC` (обычно `PR-AUC` заметно ниже) при заданном сильном дисбалансе; во второй — `PSI`, посчитанный между исходным и намеренно сдвинутым (`mean=4.5` вместо `4.0` в лог-нормальном распределении) распределением сумм, должен показать **заметно повышенное** значение, попадающее в зону «следить» или «серьёзный сдвиг», в зависимости от точной величины сдвига — сверьте порядок величины с ручным расчётом раздела 7.2.

## 9. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Приведите пример, где ROC-AUC=0.93, а модель на практике бесполезна | При сильном дисбалансе классов даже большое абсолютное число ложных срабатываний (`FP`) остаётся малой **долей** от огромного пула честных объектов (`TN`), что удерживает `FPR` низким и ROC-AUC — высоким, тогда как Precision (сравнивающая `FP` напрямую с малочисленным `TP`) может оказаться катастрофически низкой |
| Почему AUC-метрики не говорят ничего о калибровке модели? | Оба AUC зависят только от относительного ранжирования скоров; любое строго монотонное преобразование скоров не меняет AUC, но может произвольно исказить абсолютные значения вероятности |
| В чём разница между Data Drift и Concept Drift? | Data Drift — изменилось распределение входных признаков `P(X)` при неизменной истинной зависимости `P(Y|X)`; Concept Drift — изменилась сама зависимость `P(Y|X)`, те же признаки теперь значат другое |
| Почему Concept Drift опаснее Data Drift на практике? | Переобучение на свежих данных естественно устраняет Data Drift; для устранения Concept Drift нужны свежие **размеченные** примеры новой закономерности, которых часто мало, плюс типична задержка разметки (label latency), из-за чего проблема успевает нанести ущерб до того, как будет официально подтверждена |
| Что означает PSI=0.3? | Значение выше порога 0.25 — серьёзный сдвиг распределения, требующий активного пересмотра модели (переобучение и/или расследование причины сдвига), а не просто пассивного наблюдения |

## 10. Чек-поинт — попробуйте ответить без подсказок

1. Приведите пример, где ROC-AUC = 0.93, а модель на практике бесполезна — почему так может быть?
2. Что означает PSI = 0.3 и что с этим нужно делать?
3. Почему монотонное преобразование предсказанных скоров не меняет ROC-AUC и PR-AUC, но может полностью сломать бизнес-формулу вроде Cost-Sensitive Threshold?
4. Почему Isotonic Regression требует больше калибровочных данных, чем Platt Scaling, и как это связано с общей темой курса?
5. Почему мониторинг PSI без меток особенно ценен именно для детекции Concept Drift, а не только Data Drift?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. При сильном дисбалансе классов знаменатель `FPR=FP/(FP+TN)` доминируется огромным `TN` — даже большое абсолютное число ложных срабатываний (например, `500`) остаётся малой относительной долей (`~5%`) от тысяч честных объектов, что удерживает ROC-кривую высокой на всём диапазоне разумных порогов. При этом Precision, сравнивающая то же самое `FP` напрямую с малочисленным `TP` (например, `18`), может составить всего `3-4%` — операционно означая, что подавляющее большинство сигналов тревоги ложные, что делает модель практически непригодной для реального использования, несмотря на высокий ROC-AUC.

2. `PSI=0.3` превышает порог `0.25`, что классифицируется как серьёзный сдвиг распределения между эталонным (обычно train) и текущим (продакшн) распределением признака или выходного скора модели. Необходимое действие — не просто продолжать наблюдение, а активно реагировать: как минимум запланировать переобучение модели на свежих данных, как максимум — расследовать причину сдвига, чтобы отличить безобидный Data Drift от потенциально более опасного Concept Drift.

3. Оба AUC вычисляются исключительно на основе **относительного порядка** предсказанных скоров (кто выше, кто ниже) — они математически не используют абсолютные значения чисел вообще. Монотонное преобразование по определению сохраняет относительный порядок, поэтому AUC остаётся неизменным. Но формула Cost-Sensitive Threshold использует **абсолютное** значение предсказанной вероятности напрямую (`FN × TransactionAmt` домножается на вероятность через выбор порога `τ`, применяемого к реальному числовому значению `p`) — если это число искажено (пусть и монотонно), бизнес-расчёт ожидаемых издержек будет систематически неверным, даже если ранжирование объектов по риску остаётся идеальным.

4. Isotonic Regression — непараметрический метод: он не предполагает заранее заданную (сигмоидную) форму искажения калибровки, а подбирает произвольную монотонную ступенчатую функцию — это даёт больше гибкости (потенциально ниже bias калибровки), но и больше степеней свободы, которые нужно надёжно оценить по данным, отсюда большая потребность в объёме калибровочной выборки, чтобы не переобучиться на её шуме. Это прямое проявление того же bias-variance компромисса, который сопровождал весь курс — от глубины одиночного дерева (Модуль 1) до числа итераций бустинга (Модуль 5) и структуры симметричных деревьев CatBoost (Модуль 8): более гибкая модель (в данном случае — функция калибровки) снижает смещение, но требует больше данных, чтобы не заплатить за это ростом дисперсии.

5. Мониторинг PSI не требует размеченных данных (истинных меток `y`) — он сравнивает только распределения **признаков** (или выходного скора). Concept Drift, в отличие от Data Drift, по определению может произойти **без** заметного изменения распределения входных признаков `P(X)` — то же самое распределение `X`, но с изменившейся истинной зависимостью `P(Y|X)`, из-за чего PSI по отдельным признакам может **не** заметить Concept Drift напрямую. Однако PSI, посчитанный по **распределению самого выходного скора модели**, косвенно способен отловить и Concept Drift тоже: если модель продолжает получать похожие по распределению признаки, но реальная связь с таргетом изменилась, накопление ошибок часто постепенно проявляется как сдвиг в распределении предсказанных вероятностей (модель начинает систематически более уверенно или менее уверенно оценивать один и тот же тип объектов) — это не заменяет полноценный мониторинг качества по меткам (когда они наконец появятся), но даёт более раннее, хотя и косвенное, предупреждение, не дожидаясь задержки разметки.

</details>

## Модуль 13 закрывает теоретическую часть курса

Все 13 модулей пройдены — от анатомии одного дерева решений (Модуль 1) до мониторинга задеплоенной модели в проде (этот модуль). Следующий шаг — **Капстоун**: применение всего курса целиком в реализации Дня 1 проекта FraudGuard, где каждый инструмент из этих 13 модулей займёт своё конкретное место в едином пайплайне — от EDA до `/health`-эндпоинта с PSI-мониторингом.